In [ ]:
%matplotlib inline

import json
import os
import shutil
from glob import glob
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from loguru import logger
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor
from tqdm import tqdm
from ultralytics import YOLO

In [ ]:
# GLOBAL PARAMS
images_csv = 'wildlife-insights_4fc92e40-94fe-48d3-a51f-5ac1a03f5e03_project-2007160_data/images_2007160.csv'
deployments_csv = 'wildlife-insights_4fc92e40-94fe-48d3-a51f-5ac1a03f5e03_project-2007160_data/deployments.csv'
pickup_date = '2024-03-23'
dataset_name = 'pilot_mnt_data'
device = 'cuda'
cls_model_weights = 'yolov8x.pt'

In [ ]:
def change_file_time(file, t):
    timestamp_str = str(t)
    parsed_time = time.strptime(timestamp_str, '%Y-%m-%d %H:%M:%S')
    ts = time.mktime(parsed_time)
    os.utime(file, (ts, ts))

In [ ]:
def visualize_contours(image, binary_mask):
    contours, _ = cv2.findContours(binary_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contour_image = image.copy()
    cv2.drawContours(contour_image, contours, -1, (255,0,0), 3)
    plt.imshow(contour_image)
    plt.axis('off')
    plt.show()

In [ ]:
def calculate_area(bbox):
    xmin, ymin, xmax, ymax = bbox
    width = xmax - xmin
    height = ymax - ymin
    area = width * height
    return area

In [ ]:
def get_mask_image(im_path, bbox, show=False):
    # results = model(im_path, classes=[0])
    # boxes = results[0].boxes.xyxy.tolist()
    # if not boxes:
    #     logger.error(f'Could not find a human in {im_path}!')
    #     return im_path
    # if len(boxes) > 1:
    #     bbox1_area = calculate_area(boxes[0])
    #     bbox2_area = calculate_area(boxes[1])
    #     if bbox1_area > bbox2_area:
    #         idx = 1
    #     elif bbox2_area > bbox1_area:
    #         idx = 0
    # else:
    #     idx = 0

    # if im_path in switch_index:
    #     idx = idx + 1 if idx == 0 else idx - 1
    #     if len(boxes) != 2:
    #         logger.error(f'Could not find calib-human in {im_path}!')
    #         return im_path

    # bbox = boxes[0]


    image = cv2.cvtColor(cv2.imread(im_path), cv2.COLOR_BGR2RGB)
    predictor.set_image(image)
    
    input_box = np.array(bbox)
    masks, _, _ = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=input_box,
        multimask_output=False,
    )

    segmentation_mask = masks[0]
    
    # Convert the segmentation mask to a binary mask
    binary_mask = np.where(segmentation_mask > 0.5, 1, 0)
    # binary_mask = debug_fill_mask(binary_mask)
    visualize_contours(image, binary_mask)
    # binary_mask = fill_mask(binary_mask) # <<<<<<<<<<<<
    
    # Create white background and black background
    white_background = np.ones_like(image) * 255
    black_background = np.zeros_like(image)
    
    # Apply the binary mask to the white background
    new_image = white_background * binary_mask[..., np.newaxis] + black_background * (1 - binary_mask[..., np.newaxis])
    
    # Display the image
    plt.imsave(f'{deployment_path}/calibration_frames_masks/{Path(im_path).name}', new_image.astype(np.uint8))
    if show:
        plt.imshow(new_image.astype(np.uint8))
        plt.axis('off')
        plt.show()

In [ ]:
def get_bbox_image(im_path, bbox):
    # results = model.predict(source=im_path, classes=[0])
    # boxes = results[0].boxes.xyxy.tolist()
    # if not boxes:
    #     logger.error(f'Could not find a human in {im_path}!')
    #     return im_path
    # if len(boxes) > 1:
    #     bbox1_area = calculate_area(boxes[0])
    #     bbox2_area = calculate_area(boxes[1])
    #     if bbox1_area > bbox2_area:
    #         idx = 1
    #     elif bbox2_area > bbox1_area:
    #         idx = 0
    # else:
    #     idx = 0

    # if im_path in switch_index:
    #     idx = idx + 1 if idx == 0 else idx - 1
    #     if len(boxes) != 2:
    #         logger.error(f'Could not find calib-human in {im_path}!')
    #         return im_path

    # bbox = boxes[idx]

    image = cv2.imread(im_path)
    mask = np.zeros_like(image)
    x1, y1, x2, y2 = [int(coord) for coord in bbox]
    mask[y1:y2, x1:x2] = [255, 255, 255]
    output_image_path = f'{deployment_path}/calibration_frames_masks/{Path(im_path).name}'
    cv2.imwrite(output_image_path, mask)

    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 3)
    # Convert BGR to RGB for matplotlib display since OpenCV uses BGR by default.
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Display the image inline in Jupyter.
    plt.figure(figsize=(10, 10))  # You can adjust the figure size as needed.
    plt.imshow(image_rgb)
    plt.axis('off')  # Turn off axis numbers and ticks.
    plt.show()

In [ ]:
df = pd.read_csv(images_csv)
print('df length:', len(df))
print('Number of deployments in df:', len(df['deployment_id'].unique()))

df_deployments = pd.read_csv(deployments_csv)
df = df.merge(df_deployments[['camera_name', 'deployment_id']], on='deployment_id', how='inner')

df['deployment_id'] = df['deployment_id'].str.replace('/', '-')
df['timestamp'] = pd.to_datetime(df['timestamp'])

df_animals = df[~df['common_name'].isin(['Human-Camera Trapper', 'Human-Faces', 'Human', 'Blank'])]
print('Animals df length:', len(df_animals))

df_animals = df_animals.sort_values(by='timestamp')
df_animals.reset_index(drop=True, inplace=True)

df_animals['location'].to_csv('animal_images.csv', index=False, header=False)
Path('animal_images').mkdir(exist_ok=True)

# ! cat 'animal_images.csv' | gsutil -m cp -I 'animal_images'

for x in df_animals['deployment_id'].unique():
    Path(f'animal_images/{x}').mkdir(exist_ok=True, parents=True)

for x, y, z, t in zip(df_animals['location'], df_animals['deployment_id'], df_animals['image_id'], df_animals['timestamp']):
    out_path = f'animal_images/{y}/{z}{Path(x).suffix}'
    shutil.move(f'animal_images/{Path(x).name}', out_path)
    change_file_time(out_path, t)


In [ ]:
print(f'Cameras pick-up date: {pickup_date}')
df_human = df[df['timestamp'].dt.date == pd.to_datetime(pickup_date).date()]

df_human = df_human[df_human['common_name'].isin(['Human-Faces'])]
print('Animals df length:', len(df_human))

df_human['location'].to_csv('human_images.csv', index=False, header=False)
Path('human_images').mkdir(exist_ok=True)

# ! cat 'human_images.csv' | gsutil -m cp -I 'human_images'

for x in df_human['deployment_id'].unique():
    Path(f'human_images/{x}').mkdir(exist_ok=True, parents=True)

for x, y, z, t in zip(df_human['location'], df_human['deployment_id'], df_human['image_id'], df_human['timestamp']):
    out_path = f'human_images/{y}/{z}{Path(x).suffix}'
    shutil.move(f'human_images/{Path(x).name}', out_path)
    change_file_time(out_path, t)


In [ ]:
human = glob('human_images/[!_]*')
animal = glob('animal_images/[!_]*')

h = [Path(x).name for x in human]
a = [Path(x).name for x in animal]
print([x for x in a if x not in h])
assert not [x for x in a if x not in h]

#---------------------------------------------------
# WARNING !
# SPECIFIC TO ONE DATASET, DON'T USE IN PRODUCTION
mapping = {
    'HC500 Hyperfire Semi-Covert IR': 'Reconyx HC500 Hyperfire',
    'HC500 Hyperfire Semi-Covert IR_': 'Reconyx HC500 Hyperfire'
}
df['camera_name'] = df['camera_name'].replace(mapping)
# WARNING !
#---------------------------------------------------

for camera_name in df['camera_name'].unique():
    Path(f'{dataset_name}/{camera_name}/transects').mkdir(exist_ok=True, parents=True)
    Path(f'{dataset_name}/{camera_name}/results').mkdir(exist_ok=True)

for deployment in h:
    camera_name = df[df['deployment_id'] == deployment]['camera_name'].unique()[0]
    for subfolder in ['calibration_frames', 'calibration_frames_masks', 'detection_frames']:
        Path(f'{dataset_name}/{camera_name}/transects/{deployment}/{subfolder}').mkdir(exist_ok=True, parents=True)


for deployment in human:
    deployment_name = Path(deployment).name
    camera_name = df[df['deployment_id'] == deployment_name]['camera_name'].unique()[0]
    for image in glob(f'{deployment}/*'):
        out_path = f'{dataset_name}/{camera_name}/transects/{Path(deployment).name}/calibration_frames'
        shutil.copy2(image, out_path)

for deployment in animal:
    deployment_name = Path(deployment).name
    camera_name = df[df['deployment_id'] == deployment_name]['camera_name'].unique()[0]
    for image in glob(f'{deployment}/*'):
        out_path = f'{dataset_name}/{camera_name}/transects/{Path(deployment).name}/detection_frames'
        shutil.copy2(image, out_path)


In [ ]:
switch_index = ['pilot_mnt_data/Reconyx Hyperfire 2/transects/PM3 02-10-2024/calibration_frames/18.JPG',
 'pilot_mnt_data/Reconyx Hyperfire 2/transects/PM3 02-10-2024/calibration_frames/21.JPG',
 'pilot_mnt_data/Reconyx Hyperfire 2/transects/PM3 02-10-2024/calibration_frames/24.JPG',
 'pilot_mnt_data/Reconyx Hyperfire 2/transects/Cam23/calibration_frames/15.JPG',
 'pilot_mnt_data/Reconyx Hyperfire 2/transects/Cam23/calibration_frames/21.JPG',
 'pilot_mnt_data/Reconyx Hyperfire 2/transects/cam12 02-10-2024/calibration_frames/09.JPG',
 'pilot_mnt_data/Reconyx Hyperfire 2/transects/PM6 02-10-2024/calibration_frames/21.JPG',
 'pilot_mnt_data/Browning Recon Force Elite/transects/PM6 02-10-2024/calibration_frames/21.JPG',
 'pilot_mnt_data/Browning Recon Force Elite/transects/PM3 02-10-2024/calibration_frames/18.JPG',
 'pilot_mnt_data/Browning Recon Force Elite/transects/PM3 02-10-2024/calibration_frames/21.JPG',
 'pilot_mnt_data/Browning Recon Force Elite/transects/PM3 02-10-2024/calibration_frames/24.JPG']

In [ ]:
transects = glob(f'{dataset_name}/*/transects/*')
print(len(transects))
transects

In [ ]:
transects = glob(f'{dataset_name}/*/transects/*')

model = YOLO(cls_model_weights)

sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
predictor = SamPredictor(sam)

In [ ]:
with open('boxes.json') as j:
    ls_data = json.load(j)

In [ ]:
def ls_to_yolo(bbox):
    x_min = int(bbox['x'] / 100 * bbox['original_width'])
    y_min = int(bbox['y'] / 100 * bbox['original_height'])
    x_max = int((bbox['x'] + bbox['width']) / 100 * bbox['original_width'])
    y_max = int((bbox['y'] + bbox['height']) / 100 * bbox['original_height'])
    return [x_min, y_min, x_max, y_max]

In [ ]:
dply_names = [Path(x).name for x in transects]
dply_dict = {x: [] for x in dply_names}
for x in ls_data:
    dply_name = Path(x['image']).parent.name
    n = Path(x['image']).stem
    x = x['label'][0]
    _d = {'x': x['x'], 'y': x['y'], 'width': x['width'], 'height': x['height'], 'original_width': x['original_width'], 'original_height': x['original_height']}
    d = {n: _d}
    dply_dict[dply_name].append(d)

In [ ]:
dply_dict

In [ ]:
dply_dict

In [ ]:
# SAM MASKS

failed_images = []

for deployment_path in transects:
    ims = glob(f'{deployment_path}/calibration_frames/*')
    ims.sort()
    for im_path in ims:
        dply = Path(im_path).parent.parent.name
        dply_boxes = dply_dict[dply]
        n = Path(im_path).stem
        _bbox = [x for x in dply_boxes if x.get(n)][0][n]
        bbox = ls_to_yolo(_bbox)
        failed_image_path = get_mask_image(im_path, bbox, show=False)
        if failed_image_path:
            logger.warning(f'Failed to extract mask from: {failed_image_path}')
            # Path(im_path).unlink()
            failed_images.append(im_path)

In [ ]:
## BBOX MASKS

failed_images = []

for deployment_path in transects:
    ims = glob(f'{deployment_path}/calibration_frames/*')
    ims.sort()
    for im_path in ims:
        dply = Path(im_path).parent.parent.name
        dply_boxes = dply_dict[dply]
        n = Path(im_path).stem
        _bbox = [x for x in dply_boxes if x.get(n)][0][n]
        bbox = ls_to_yolo(_bbox)
        failed_image_path = get_bbox_image(im_path, bbox)
        if failed_image_path:
            logger.warning(f'Failed to extract mask from: {failed_image_path}')
            # Path(im_path).unlink()
            failed_images.append(im_path)

In [ ]:
# failed_images += [x.replace('calibration_frames', 'calibration_frames_masks') for x in failed_images]

# for x in failed_images:
#     os.remove(x)


failed_images

In [ ]:
for camera_name in df['camera_name'].unique():
    parents = glob(f'{dataset_name}/{camera_name}/transects/*')
    for x in parents:
        l1 = len(glob(f'{x}/calibration_frames/*'))
        l2 = len(glob(f'{x}/calibration_frames_masks/*'))
        assert l1 == l2

In [ ]:
with open('cameras_specs.json') as j:
    camera_specs = json.load(j)

In [ ]:
project_per_camera = glob(f'{dataset_name}/*')
project_per_camera

In [ ]:
camera_specs

```bash
export LD_LIBRARY_PATH=/home/biodiv/mambaforge/envs/dnd/lib/python3.11/site-packages/nvidia/cudnn/lib:$LD_LIBRARY_PATH
```

```bash
python main.py \
    --cli \
    --bbox_confidence_threshold 0.50 \
    --bbox_sampling_percentile 50 \
    --calibration_regression_method "ransac" \
    --camera_horizontal_fov 38.2 \
    --camera_vertical_fov 29.1 \
    --detection_sampling_method "sam" \
    --max_depth 24 \
    --min_depth 1 \
    --sample_from "detection" \
    --data_dir "../pilot_mnt_data/Reconyx Hyperfire 2" \
    --draw_world_position \
    --calibrate_metric \
    --draw_detection_ids \
    --multiple_animal_reduction only_centermost
```

```bash
python main.py \
    --cli \
    --bbox_confidence_threshold 0.50 \
    --bbox_sampling_percentile 50 \
    --calibration_regression_method "ransac" \
    --camera_horizontal_fov 42.2 \
    --camera_vertical_fov 32.3 \
    --detection_sampling_method "sam" \
    --max_depth 24 \
    --min_depth 1 \
    --sample_from "detection" \
    --data_dir "../pilot_mnt_data/Reconyx HC500 Hyperfire" \
    --draw_world_position \
    --calibrate_metric \
    --draw_detection_ids \
    --multiple_animal_reduction only_centermost
```

```bash
python main.py \
    --cli \
    --bbox_confidence_threshold 0.50 \
    --bbox_sampling_percentile 50 \
    --calibration_regression_method "ransac" \
    --camera_horizontal_fov 41.0 \
    --camera_vertical_fov 41.0 \
    --detection_sampling_method "sam" \
    --max_depth 24 \
    --min_depth 1 \
    --sample_from "detection" \
    --data_dir "../pilot_mnt_data/Browning Recon Force Elite" \
    --draw_world_position \
    --calibrate_metric \
    --draw_detection_ids \
    --multiple_animal_reduction only_centermost
```


In [ ]:
for camera_name in df['camera_name'].unique():
    results_df = pd.read_csv(f'{dataset_name}-exp4/{camera_name}/results/results.csv')
    deployment_ids = df['deployment_id'].unique()
    results_df['image_id'] = results_df['frame_id']

    results_df = pd.merge(df, results_df, on='image_id', how='inner')
    results_df.to_csv(f'{dataset_name}-exp4/{camera_name}/results/results-{camera_name}-full.csv', index=False)
